# Immo Eliza scraping notebook

This notebook uses Selenium (and sometimes BeautifulSoup) to open specific ImmoVlan pages with houses.Each code section focuses on extracting one field from a page.


### Extract property ID from URL
Opens a page with Selenium, reads the current URL, and extracts the property ID using a regex. Closes the browser afterwards.


In [1]:
from selenium import webdriver
import re

driver = webdriver.Chrome()

driver.get("https://immovlan.be/en/projectdetail/1485771-01267921_om_272263")

current_url = driver.current_url
print("Current URL:", current_url)

match = re.search(r'projectdetail/([^/?#]+)', current_url)
if match:
    property_id = match.group(1)
    print("Property ID:", property_id)
else:
    print("No match found.")

driver.quit()


Current URL: https://immovlan.be/en/projectdetail/1485771-01267921_om_272263
Property ID: 1485771-01267921_om_272263


I can also scrape with javascript

### Read IMMOVLAN_REFERENCE via JavaScript
Opens the project detail page and executes javascript to read `window.IMMOVLAN_REFERENCE` directly from the page, then prints it and quits the browser.


In [2]:
from selenium import webdriver

# Initialize Chrome WebDriver
driver = webdriver.Chrome()

# Open the property page
driver.get("https://immovlan.be/en/projectdetail/1485771-01267921_om_272263")

# Execute JS to read the variable from the window object
immovlan_ref = driver.execute_script("return window.IMMOVLAN_REFERENCE;")

print("IMMOVLAN_REFERENCE:", immovlan_ref)

driver.quit()


IMMOVLAN_REFERENCE: 1485771-01267921_OM_272263


### Extract postal code and locality
Loads the page, locates the `span.city-line` text (e.g., "2430 Laakdal"), then splits it into postal code and locality and prints both.


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By

# Initialize WebDriver
driver = webdriver.Chrome()

driver.get("https://immovlan.be/en/projectdetail/1485771-01267921_om_272263")

city_line = driver.find_element(By.CSS_SELECTOR, "span.city-line").text
print("Full city line:", city_line)

# Split postal code and locality
parts = city_line.split(" ", 1)  # Split only on the first space
if len(parts) == 2:
    postal_code, locality = parts
else:
    postal_code, locality = parts[0], ""

print("Postal code:", postal_code)
print("Locality:", locality)

driver.quit()


Full city line: 2430 Laakdal
Postal code: 2430
Locality: Laakdal


### Parse property type from a detail URL string
Takes a sample detail URL and uses a regex to extract the property type segment (e.g., `residence`).


In [4]:
url = "https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113"

match = re.search(r'detail/([^/]+)/', url)
if match:
    property_type = match.group(1)
    print("Property type:", property_type)
else:
    print("No match found.")


Property type: residence


### Scrape "State of the property"
Looks for the `h4` labeled "State of the property", then prints the `<p>` text.


In [5]:
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()

driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

driver.implicitly_wait(5)

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

h4 = soup.find("h4", string=lambda t: t and "State of the property" in t)
if h4:
    p_tag = h4.find_next("p")
    if p_tag:
        print("State of the property:", p_tag.get_text(strip=True))
    else:
        print("No <p> tag found after the h4.")
else:
    print("No h4 found with that label.")

driver.quit()



State of the property: Fully renovated


### Extract livable surface (m²)
Finds the "Livable surface" section, reads the adjacent text, and gets the numeric value using regex.


In [6]:
from selenium import webdriver
from bs4 import BeautifulSoup
import re

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

soup = BeautifulSoup(driver.page_source, "html.parser")

# Find the <h4> label
h4 = soup.find("h4", string=lambda t: t and "Livable surface" in t)
if h4:
    # Get the <p> text next to it
    p_text = h4.find_next("p").get_text(strip=True)
    
    # Extract the number before 'm²' using regex
    match = re.search(r"(\d+)", p_text)
    if match:
        livable_surface = int(match.group(1))
        print("Livable surface:", livable_surface)
    else:
        print("No number found.")
else:
    print("Couldn't find the 'Livable surface' section.")

driver.quit()


Livable surface: 127


### Determine if the kitchen is fully equipped
Finds the "Kitchen equipment" section, reads the text, and returns a boolean indicating if it equals "Fully equipped".


In [7]:
from selenium import webdriver
from bs4 import BeautifulSoup

# open the page
driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

# parse page source
soup = BeautifulSoup(driver.page_source, "html.parser")

h4 = soup.find("h4", string="Kitchen equipment")

if h4:
    text = h4.find_next("p").get_text(strip=True)
    kitchen_equipped = text == "Fully equipped"
    print(kitchen_equipped)
else:
    print("Kitchen equipment not found")

driver.quit()


True


### Determine if there is a terrace
Finds the "Terrace" section and returns `True` if the adjacent text equals "Yes" (case-insensitive), otherwise `False`.


In [8]:
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

soup = BeautifulSoup(driver.page_source, "html.parser")

h4 = soup.find("h4", string="Terrace")

if h4:
    text = h4.find_next("p").get_text(strip=True)
    has_terrace = text.lower() == "yes"
    print(has_terrace)
else: 
    has_terrace = False
    print("Terrace not found")

driver.quit()


True


### Parse number of facades
Looks up the "Number of facades" heading, reads the next `<p>` text, and converts it to an integer if possible.


In [9]:
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

soup = BeautifulSoup(driver.page_source, "html.parser")

# Find the <h4> with "Number of facades"
h4 = soup.find("h4", string="Number of facades")

if h4:
    text = h4.find_next("p").get_text(strip=True)
    try:
        number_of_facades = int(text)
    except ValueError:
        number_of_facades = None
    print(number_of_facades)
else:
    print("Number of facades not found")

driver.quit()


4


### Determine if there is a fireplace
Finds the "Fireplace" section and returns `True` if the text equals "Yes", otherwise `False`.


In [10]:
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

soup = BeautifulSoup(driver.page_source, "html.parser")

h4 = soup.find("h4", string="Fireplace")

if h4:
    text = h4.find_next("p").get_text(strip=True)
    has_fireplace = text.lower() == "yes"
    print(has_fireplace)
else:
    print("Fireplace not found")

driver.quit()


True


### Determine if there is a swimming pool
Finds the "Swimming pool" section and returns `True` if the text equals "Yes", otherwise `False`.


In [11]:
from selenium import webdriver
from bs4 import BeautifulSoup

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/6830/bouillon/rwc41113")

soup = BeautifulSoup(driver.page_source, "html.parser")

# Find the <h4> with text "Swimming pool"
h4 = soup.find("h4", string="Swimming pool")

if h4:
    text = h4.find_next("p").get_text(strip=True)
    has_pool = text.lower() == "yes"
    print(has_pool)
else:
    print("Swimming pool not found")
    has_pool = False

driver.quit()


False


### Extract terrace surface (m²)
Find the <h4> with text "Surface terrace" (allow partial match in case of formatting differences)

In [17]:
from selenium import webdriver
from bs4 import BeautifulSoup
import re

driver = webdriver.Chrome()
driver.get("https://immovlan.be/en/detail/residence/for-sale/4520/vinalmont/vwd15444")

soup = BeautifulSoup(driver.page_source, "html.parser")

h4 = soup.find("h4", string="Surface terrace")

surface_terrace = None
if h4:
    p_tag = h4.find_next("p")
    if p_tag:
        text = p_tag.get_text(strip=True)
        match = re.search(r"\d+", text)
        if match:
            surface_terrace = int(match.group())
    print("Surface terrace (m²):", surface_terrace)
else:
    surface_terrace = False
    print("Surface terrace not found")

driver.quit()


Surface terrace (m²): 70


We also tried to see what data every property contains, so we thought to put it in a class. Later on we decided to put it in a dict actually, which we read out to append to the csv... So we didn't used this class. But it was need to see the types of every variable...

In [ ]:
class Property():
    """
    Class to store data describing a property to buy
    """

    def __init__(self, id, locality: str = None, postcode: int = None, price: int = None, type: int = 0, subtype: str = None, sale_type: int = 0, nb_rooms: int = None, area: int = None, equipped_kitchen: int = None, furnished: int = None, open_fire: int = None, terrace: int = None, garden: int = None, facades: int = None, pool: int= None, state: int = 0,url: str = None):
        self.id = id
        self.locality = locality
        self.postcode = postcode
        self.price = price
        self.type = type # (house - 0/ appartment - 2)
        self.subtype = subtype # property subtype
        self.sale_type = sale_type # type of sale (1 - to rent / 2 - for sale / 3 - public sale / 4 - shared accomodation) (no life sales)
        self.rooms = nb_rooms #number of rooms
        self.area = area #living area in m2
        self.kitchen = equipped_kitchen # (No - 0/ Yes - 1)
        self.furnished = furnished # (No - 0/ Yes - 1)
        self.open_fire = open_fire # (No - 0/ Yes - 1), open fire presence
        self.terrace = terrace # area in m2 or None if no terrace
        self.garden = garden # area in m2 or None if no garden
        self.facades = facades # number of facades
        self.pool = pool # (No - 0/ Yes - 1) presence or swimming pool
        self.state = state # state of the house (1 - new / 2 - excellent / 3 - fully renovated / 4 - normal / 5 - to renovate)
       # self.energy_class = energy_class # state of the house (1 - new / 2 - excellent / 3 - fully renovated / 4 - normal / 5 - to renovate)
        self.url = url